# Coding Assignment 1: Household support

**Research Methods 2026** · Individual · 100 marks · approximately three hours

This notebook follows the five questions in main.tex. Complete each written answer and every TODO. Before submission, save as **exercise_01.ipynb** (the filename required by the instructions), include README.md, then restart the kernel and run all cells in order.

Use basic Python, conditions, loops, functions, lambda, built-ins, and math.isclose. No CSV, pandas, NumPy, plotting, list comprehensions, or try/except are required. AI assistance is permitted; record your use in the README.


## Supplied household inputs and policy

List positions refer to the same household. Income and consumption are EUR/month; size is persons. Preserve these lists. For income I, consumption C, size n: if I/n > 600, transfer T=0; otherwise T=min(100*n, max(C-I, 0)). Exactly EUR 600 per person is eligible. Income after support Y=I+T; saving before S=I-C; saving after S_after=Y-C. Share is saving/income, or None when income is zero. Negative saving is valid.


In [1]:
import math

household_ids = ["A01", "A02", "A03", "A04",
                 "A05", "A06", "A07", "A08"]
income_text = ["1200", "0", "1500", "700",
               "2400", "800", "0", "1800"]
consumptions = [1250, 150, 1600, 850, 2600, 950, 0, 1400]
household_sizes = [2, 1, 3, 1, 4, 2, 2, 3]


## Question 1 — Trace values and notebook state (12 marks)

Before running the snippet, write your predictions below and keep that first attempt. Then run the snippet and explain corrections. (a) Give the three output lines and trace relevant values (4). (b) Explain why appending through saved_incomes affects incomes, and why changing income does not update first_share (4). (c) Make an independent copy of the original list, change income to 1200, recalculate the share, and show the new three output lines (4).


**A) Original prediction:**

[1200, 800, 0, 600]

600 4

20%

Incomes is saved as saved_incomes therefore any mention of either of those variables will change the value of the other. Incomes[0] which is the first item on the list is changed to 1200. Additionally, saved_incomes is appended and 600 is added to the end of the list. This also changes the list saved to the variable incomes. That is why when incomes is printed the list is such as provided above.

saved_incomes[-1] indicates the last item on the list. Since saved_incomes is appended before, the last item is 600. The length of incomes is then requested, since incomes=saved_incomes the length is 4 as there is one append that has been done as previously mentioned.

Because first_share is calculated before incomes[0] is changed to 1200, the share is 20%. It is called by first_share:.0% as a percentage value without any decimal places.

**B) Explanation, and corrections:**

All of this was answered above due to tracing values, but saved_incomes=incomes makes it so both values are updated when one is updated. Additionally, incomes[0] is changed after first_share is calculated therefore first share does not update unless it is run AFTER the incomes[0] is changed.


In [2]:
incomes = [1000, 800, 0]
saved_incomes = incomes
first_share = (incomes[0] - 800) / incomes[0]
incomes[0] = 1200
saved_incomes.append(600)
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")


[1200, 800, 0, 600]
600 4
20%


### Question 1(c) — Independent copy

Use code to recreate the original list, make an independent copy, change its first income, recalculate the share, and print the requested three output lines.


In [3]:
incomes = [1200, 800, 0]
saved_incomes = incomes
first_share = (incomes[0] - 800) / incomes[0]
incomes[0] = 1200
saved_incomes.append(600)
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0, 600]
600 4
33%


## Question 2 — Implement the policy as functions (28 marks)

Implement these functions; return values rather than only printing. Assume finite real amounts and integer sizes; no general type/NaN/infinity checks.

- saving_share(income, consumption) (8): raise ValueError for either negative amount, checking before zero income. Return None for zero income, otherwise (income-consumption)/income as a fraction.
- transfer(income, consumption, size) (12): raise ValueError for negative income/consumption or size <= 0, before eligibility. Implement the exact policy; every valid call returns a number including zero.
- post_policy_share(income, consumption, size) (8): call transfer, add it to original income, then call saving_share with new income and unchanged consumption. Reuse both helpers.

Explain in at most 100 words beside the code: parameters vs arguments, why return is needed, and why checks precede division. Functions must use their arguments, not notebook values.


In [4]:
# Parameters are the input names and arguments are the actual values passed when calling a function.
# The functions use these arguments so they work for different households.
def saving_share(income, consumption):
    # Check invalid negative amounts first and handle zero before dividing by income otherwise you get an error.
    if income < 0 or consumption < 0:
        raise ValueError("income and consumption must be non-negative")
    if income == 0:
        return None
    # return sends the calculated result back to the code that called this function rather than printing so that it can be used further.
    return (income - consumption) / income


def transfer(income, consumption, size):
    if income < 0 or consumption < 0 or size <= 0:
        raise ValueError("income and consumption must be non-negative, and size must be positive")
    if income / size > 600:
        return 0
    return min(100 * size, max(consumption - income, 0))


def post_policy_share(income, consumption, size):
    support = transfer(income, consumption, size)
    income_after_support = income + support
    return saving_share(income_after_support, consumption)






## Question 3 — Build and interpret a checked report (30 marks)

Use your functions. Use explicit loops where requested, keep full precision, and round only for display. State the population/denominator for counts.


### 3(a) Prepare inputs (4)

Convert income_text to a new numeric list using a loop; leave the strings unchanged. Check all four input lists have equal lengths before processing. Raise ValueError on mismatch; do not let zip silently omit households.


In [5]:
# Check that the four supplied lists have equal lengths before processing.
if not (len(household_ids) == len(income_text) == len(consumptions) == len(household_sizes)):
    raise ValueError("the four household input lists must have equal lengths")

# Convert income_text into a new numeric list; keep the original strings unchanged.
incomes = []
for value in income_text:
    incomes.append(int(value))

print("Numeric incomes:", incomes)
print("Original income_text:", income_text)


Numeric incomes: [1200, 0, 1500, 700, 2400, 800, 0, 1800]
Original income_text: ['1200', '0', '1500', '700', '2400', '800', '0', '1800']


### 3(b) Calculate household results (8)

Use an explicit loop over aligned inputs. Build result lists and print one labelled line per household with ID, original income, consumption, size, transfer, saving before/after, and share before/after. Format EUR amounts and percentage displays to two decimal places; print undefined for None. Never replace undefined share with zero.


In [6]:
# Use the inputs prepared in 3(a) and functions defined in Question 2.
transfers, incomes_after, savings_before, savings_after = [], [], [], []
shares_before, shares_after = [], []

for household_id, income, consumption, size in zip(
        household_ids, incomes, consumptions, household_sizes):
    amount = transfer(income, consumption, size)
    after_income = income + amount
    before_saving = income - consumption
    after_saving = after_income - consumption
    before_share = saving_share(income, consumption)
    after_share = post_policy_share(income, consumption, size)

    transfers.append(amount)
    incomes_after.append(after_income)
    savings_before.append(before_saving)
    savings_after.append(after_saving)
    shares_before.append(before_share)
    shares_after.append(after_share)

    before_text = f"{before_share:.2%}" if before_share is not None else "undefined"
    after_text = f"{after_share:.2%}" if after_share is not None else "undefined"
    print(f"{household_id} | income €{income:.2f} | consumption €{consumption:.2f} | size {size} | "
          f"transfer €{amount:.2f} | saving before €{before_saving:.2f} | saving after €{after_saving:.2f} | "
          f"share before {before_text} | share after {after_text}")


A01 | income €1200.00 | consumption €1250.00 | size 2 | transfer €50.00 | saving before €-50.00 | saving after €0.00 | share before -4.17% | share after 0.00%
A02 | income €0.00 | consumption €150.00 | size 1 | transfer €100.00 | saving before €-150.00 | saving after €-50.00 | share before undefined | share after -50.00%
A03 | income €1500.00 | consumption €1600.00 | size 3 | transfer €100.00 | saving before €-100.00 | saving after €0.00 | share before -6.67% | share after 0.00%
A04 | income €700.00 | consumption €850.00 | size 1 | transfer €0.00 | saving before €-150.00 | saving after €-150.00 | share before -21.43% | share after -21.43%
A05 | income €2400.00 | consumption €2600.00 | size 4 | transfer €200.00 | saving before €-200.00 | saving after €0.00 | share before -8.33% | share after 0.00%
A06 | income €800.00 | consumption €950.00 | size 2 | transfer €150.00 | saving before €-150.00 | saving after €0.00 | share before -18.75% | share after 0.00%
A07 | income €0.00 | consumption

### 3(c) Compare totals and averages (8)

Calculate/report: total transfer; households with positive transfer; counts with negative saving before/after; pooled shares sum(S)/sum(I) and sum(S_after)/sum(Y); arithmetic means of defined household shares before/after and count in each mean; and mean after support restricted to households whose before-share was defined, using those same households in both periods.

In at most 120 words, compare pooled shares and household means, identify any change in which households enter the means, and explain why means across different sets can mislead. Only totals, counts, and arithmetic averages are needed.


In [7]:
# Use the aligned result lists created by the Question 3(b) cell.
household_count = len(household_ids)
total_transfer = sum(transfers)
total_income_before = sum(incomes)
total_income_after = sum(incomes_after)
total_saving_before = sum(savings_before)
total_saving_after = sum(savings_after)

positive_transfer_count = 0
negative_saving_before_count = 0
negative_saving_after_count = 0
before_share_sum = 0
before_share_count = 0
after_share_sum = 0
after_share_count = 0
same_group_after_sum = 0
same_group_count = 0

for i in range(household_count):
    if transfers[i] > 0:
        positive_transfer_count += 1
    if savings_before[i] < 0:
        negative_saving_before_count += 1
    if savings_after[i] < 0:
        negative_saving_after_count += 1

    if shares_before[i] is not None:
        before_share_sum += shares_before[i]
        before_share_count += 1
        # Include these same households in the after-support comparison.
        same_group_after_sum += shares_after[i]
        same_group_count += 1
    if shares_after[i] is not None:
        after_share_sum += shares_after[i]
        after_share_count += 1

pooled_share_before = total_saving_before / total_income_before
pooled_share_after = total_saving_after / total_income_after
mean_share_before = before_share_sum / before_share_count
mean_share_after = after_share_sum / after_share_count
mean_after_same_group = same_group_after_sum / same_group_count

print(f"Total transfer: €{total_transfer:.2f}")
print(f"Households receiving a positive transfer: {positive_transfer_count} of {household_count}")
print(f"Households with negative saving before support: {negative_saving_before_count} of {household_count}")
print(f"Households with negative saving after support: {negative_saving_after_count} of {household_count}")
print(f"Pooled saving share before support: {pooled_share_before:.2%}")
print(f"Pooled saving share after support: {pooled_share_after:.2%}")
print(f"Mean defined household share before support: {mean_share_before:.2%} ({before_share_count} households)")
print(f"Mean defined household share after support: {mean_share_after:.2%} ({after_share_count} households)")
print(f"Mean after support for households with a defined before share: {mean_after_same_group:.2%} ({same_group_count} households)")


Total transfer: €600.00
Households receiving a positive transfer: 5 of 8
Households with negative saving before support: 6 of 8
Households with negative saving after support: 2 of 8
Pooled saving share before support: -4.76%
Pooled saving share after support: 2.22%
Mean defined household share before support: -6.19% (6 households)
Mean defined household share after support: -7.03% (7 households)
Mean after support for households with a defined before share: 0.13% (6 households)


**Interpretation (120 words maximum):**

The pooled share changes from -4.76% to 2.22%, comparing total savings with total income. The mean defined household share changes from -6.19% (6 households) to -7.03% (7 households). The post-support mean newly includes A02: its before share was undefined because income was zero, but its €100 transfer makes the after share -50.00%. For the same six households, the mean changes from -6.19% to 0.13%. Comparing means over different groups can mislead because the full post-support mean includes A02. Pooled shares weight by income, while household means give each household equal weight.


### 3(d) Use an inline function (4)

Build a new list containing only the defined pre-policy household shares. Use sorted with a lambda key to order them by distance from a target share of 0.10, nearest first. Print the original share values, including their signs, not the distances. Preserve the original result list, including None. In the Markdown cell below, explain what the lambda receives and returns.


In [8]:
defined_pre_policy_shares = []
for share in shares_before:
    if share is not None:
        defined_pre_policy_shares.append(share)

shares_nearest_to_10_percent = sorted(
    defined_pre_policy_shares,
    key=lambda share: abs(share - 0.10)
)
print(shares_nearest_to_10_percent)


[0.2222222222222222, -0.041666666666666664, -0.06666666666666667, -0.08333333333333333, -0.1875, -0.21428571428571427]


**Lambda explanation:**

The lambda receives one defined pre-policy share at a time. It returns that share's absolute distance from 0.10, which sorted uses as the ordering key; the printed list still contains the original shares.


### 3(e) Test a counterfactual (6)

Make an independent copy of the numeric income list and change only A01's income from 1200 to 1202. Recompute A01's transfer, income after support, and saving after support. Show each quantity under both inputs and the change in each. Explain in at most 80 words whether higher original income necessarily gives higher income after support under this policy. Verify that the original income list remains unchanged.


In [9]:
original_incomes_before_counterfactual = incomes.copy()
counterfactual_incomes = incomes.copy()
a01_index = household_ids.index("A01")
counterfactual_incomes[a01_index] = 1202

original_a01_transfer = transfer(
    incomes[a01_index], consumptions[a01_index], household_sizes[a01_index]
)
original_a01_income_after = incomes[a01_index] + original_a01_transfer
original_a01_saving_after = original_a01_income_after - consumptions[a01_index]

counterfactual_a01_transfer = transfer(
    counterfactual_incomes[a01_index], consumptions[a01_index], household_sizes[a01_index]
)
counterfactual_a01_income_after = counterfactual_incomes[a01_index] + counterfactual_a01_transfer
counterfactual_a01_saving_after = counterfactual_a01_income_after - consumptions[a01_index]

print("A01 original income: €1200.00")
print(f"  Transfer: €{original_a01_transfer:.2f}")
print(f"  Income after support: €{original_a01_income_after:.2f}")
print(f"  Saving after support: €{original_a01_saving_after:.2f}")
print("A01 counterfactual income: €1202.00")
print(f"  Transfer: €{counterfactual_a01_transfer:.2f}")
print(f"  Income after support: €{counterfactual_a01_income_after:.2f}")
print(f"  Saving after support: €{counterfactual_a01_saving_after:.2f}")
print("Changes (counterfactual minus original):")
print(f"  Transfer: €{counterfactual_a01_transfer - original_a01_transfer:.2f}")
print(f"  Income after support: €{counterfactual_a01_income_after - original_a01_income_after:.2f}")
print(f"  Saving after support: €{counterfactual_a01_saving_after - original_a01_saving_after:.2f}")
print("Original income list unchanged:", incomes == original_incomes_before_counterfactual)


A01 original income: €1200.00
  Transfer: €50.00
  Income after support: €1250.00
  Saving after support: €0.00
A01 counterfactual income: €1202.00
  Transfer: €0.00
  Income after support: €1202.00
  Saving after support: €-48.00
Changes (counterfactual minus original):
  Transfer: €-50.00
  Income after support: €-48.00
  Saving after support: €-48.00
Original income list unchanged: True


**Counterfactual explanation (80 words maximum):**

No. At €1,200, A01 is exactly at the €600-per-person eligibility boundary and receives €50, making income after support €1,250. At €1,202, income per person is €601, so the transfer becomes zero and income after support is €1,202. The €2 increase in original income therefore causes a €48 decrease in income after support because eligibility ends at the threshold.


In [10]:
def proposed_mean(incomes, consumptions):
    total = 0
    count = 0
    for income, consumption in zip(incomes, consumptions):
        share = saving_share(income, consumption)
        if share is not None and share != 0:
            total = 0
            total = total + share
            count = count + 1
    return total / count


### 4(a) Four defects in the proposal

1. It skips shares equal to zero. Zero is a defined and valid share so it must be included in the mean.
2. It resets total to zero inside the loop. That discards the shares already accumulated, so the final total does not contain all included shares.
3. It uses zip without checking list lengths. If the lists differ in length, zip silently ignores the extra items instead of raising ValueError.
4. It always divides by count. If there are no defined shares, including empty inputs, count is zero and the function raises ZeroDivisionError instead of returning None.

### 4(b) Repair

The repaired function below uses saving_share, checks the lengths, includes defined zero and negative shares, leaves the inputs unchanged, and returns None when there are no defined shares.


In [11]:
def mean_defined_share(incomes, consumptions):
    if len(incomes) != len(consumptions):
        raise ValueError("incomes and consumptions must have equal lengths")

    total = 0
    count = 0
    for income, consumption in zip(incomes, consumptions):
        share = saving_share(income, consumption)
        if share is not None:
            total += share
            count += 1

    if count == 0:
        return None
    return total / count


### 4(c) Four diagnostic cases

| Case | Inputs: incomes, consumptions | Correct outcome | Proposal outcome | Defect exposed |
|---|---|---|---|---|
| 1 | [100, 100], [90, 100] | 0.05 | 0.1 | The proposal omits the valid zero share. |
| 2 | [100, 100], [90, 80] | 0.15 | 0.1 | Resetting total in each loop iteration loses earlier shares. |
| 3 | [100, 200], [90] | ValueError | 0.1 | zip truncates unequal inputs without reporting the mismatch. |
| 4 | [], [] | None | ZeroDivisionError | The proposal divides by zero instead of returning None when no shares are defined. |

Deliberately failing calls and their observed errors (shown here in Markdown so the notebook itself has no unhandled exception):

```text
>>> proposed_mean([], [])
ZeroDivisionError: division by zero

>>> mean_defined_share([100, 200], [90])
ValueError: incomes and consumptions must have equal lengths
```


## Question 5 — Evidence and reproducible hand-in (10 marks)

### 5(a) Assertions (6)

Include at least eight meaningful passing assertions covering ordinary saving, negative saving, zero income, eligibility boundary, transfer cap, no shortfall, all-undefined shares, and preserved original input in the counterfactual. Use is None and math.isclose. For four assertions, explain which plausible mistake each would detect.


In [12]:
# Saving shares: ordinary positive saving, negative saving, and zero income.
assert math.isclose(saving_share(1000, 800), 0.20)
assert math.isclose(saving_share(800, 950), -0.1875)
assert saving_share(0, 150) is None

# Transfer policy: exactly at the eligibility boundary, just above it, capped, and no shortfall.
assert math.isclose(transfer(1200, 1250, 2), 50)
assert math.isclose(transfer(1202, 1250, 2), 0)
assert math.isclose(transfer(0, 500, 2), 200)
assert math.isclose(transfer(1000, 800, 2), 0)

# The mean is None when every share is undefined; the counterfactual kept the original list unchanged.
assert mean_defined_share([0, 0], [0, 150]) is None
assert incomes == original_incomes_before_counterfactual

print("All 9 assertions passed.")


All 9 assertions passed.


**Four assertion explanations:**

1. The ordinary-saving assertion would catch returning a percentage such as 20 instead of a fractional share such as 0.20.
2. The negative-saving assertion would catch incorrectly clipping a household's negative saving share to zero.
3. The exact-boundary assertion would catch treating income of exactly €600 per person as ineligible.
4. The transfer-cap assertion would catch paying more than €100 per person when the consumption shortfall exceeds the cap.
